# Notebook 02 â€” Data Cleaning & Preprocessing

**Goal:** Check every discharge CSV for data-quality issues and build a clean
cycle-level dataset (one row per discharge cycle).

**Output:** `cleaned_data/processed_dataset.csv`

In [ ]:
import os, sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join('..', 'src'))
from data_loader import load_cycle_csv, get_discharge_metadata
from cleaning import check_discharge_csv, clean_discharge_csv, cleaning_report
from feature_engineering import extract_cycle_features

DATASET_DIR  = os.path.join('..', 'dataset', 'data')
CLEANED_DIR = os.path.join('..', 'cleaned_data')
os.makedirs(CLEANED_DIR, exist_ok=True)
print('Imports OK')

## 1. Load Discharge Metadata

In [ ]:
dis_meta = get_discharge_metadata()
print(f"Discharge cycles in metadata (valid capacity): {len(dis_meta)}")
print(f"Columns: {list(dis_meta.columns)}")
print()
print("Sample:")
print(dis_meta[['battery_id','cycle','filename','Capacity','ambient_temperature']].head(8).to_string())

## 2. Check Metadata Quality

In [ ]:
# Check metadata itself for issues
print("Missing values in discharge metadata:")
print(dis_meta[['battery_id','cycle','filename','Capacity','ambient_temperature']].isnull().sum())
print()
print(f"Capacity range: {dis_meta['Capacity'].min():.4f} â€” {dis_meta['Capacity'].max():.4f} Ah")
print(f"Ambient temperature values: {sorted(dis_meta['ambient_temperature'].unique())}")
print()
print("Discharge cycles per battery:")
print(dis_meta.groupby('battery_id').size().sort_values(ascending=False).to_string())

## 3. Validate Every Discharge CSV

In [ ]:
# Iterate through all discharge cycles, validate each CSV
results   = []   # list of {filename, battery_id, valid, reason}
rows_list = []   # list of feature dicts (for valid cycles)

total = len(dis_meta)
print(f"Processing {total} discharge CSVs ...")
print("(this may take ~30â€“60 seconds for ~2,800 files)")

for i, row in dis_meta.iterrows():
    fname  = row['filename']
    bid    = row['battery_id']
    cap    = row['Capacity']
    amb    = row['ambient_temperature']
    cyc    = row['cycle']

    df = load_cycle_csv(fname, DATASET_DIR)
    valid, reason = check_discharge_csv(df, fname)

    results.append({'filename': fname, 'battery_id': bid,
                    'valid': valid, 'reason': reason})

    if valid:
        df_clean = clean_discharge_csv(df)
        feat = extract_cycle_features(df_clean, cap, amb, bid, cyc)
        rows_list.append(feat)

    if (i + 1) % 500 == 0 or (i + 1) == total:
        n_valid = sum(r['valid'] for r in results)
        print(f"  Processed {i+1}/{total}  |  valid so far: {n_valid}")

print("\nDone.")

## 4. Cleaning Report

In [ ]:
cleaning_report(results)

## 5. Build Cycle-Level Dataset

In [ ]:
cycle_df = pd.DataFrame(rows_list)
print(f"Cycle-level dataset shape: {cycle_df.shape}")
print(f"Columns: {list(cycle_df.columns)}")
print()
print("Missing values:")
print(cycle_df.isnull().sum())
print()
print("Sample (first 5 rows):")
print(cycle_df.head().to_string())

In [ ]:
# Sort by battery and cycle
cycle_df = cycle_df.sort_values(['battery_id', 'cycle']).reset_index(drop=True)

# Check for duplicate (battery_id, cycle) pairs
dupes = cycle_df.duplicated(subset=['battery_id', 'cycle'])
print(f"Duplicate (battery_id, cycle) pairs: {dupes.sum()}")
if dupes.sum() > 0:
    print(cycle_df[dupes][['battery_id','cycle']].head())
    cycle_df = cycle_df.drop_duplicates(subset=['battery_id','cycle'], keep='first')
    print(f"After removing duplicates: {len(cycle_df)} rows")

In [ ]:
# Save
out_path = os.path.join(CLEANED_DIR, 'processed_dataset.csv')
cycle_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {cycle_df.shape}")
print(f"Size : {os.path.getsize(out_path)/1024:.1f} KB")

## Summary

In [ ]:
n_valid   = sum(r['valid'] for r in results)
n_invalid = len(results) - n_valid
print("=" * 50)
print("  PREPROCESSING SUMMARY")
print("=" * 50)
print(f"  Discharge cycles checked  : {len(results)}")
print(f"  Valid cycles kept         : {n_valid}")
print(f"  Invalid cycles removed    : {n_invalid}")
print(f"  Batteries                 : {cycle_df['battery_id'].nunique()}")
print(f"  Final rows (ML samples)   : {len(cycle_df)}")
print(f"  Final columns             : {len(cycle_df.columns)}")
print("=" * 50)
print()
print("Next: Notebook 03 â€” EDA")